In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-classic-algorithms",
    notebook_name="03_q_learning_experiments.ipynb"
)

# Experiments: Q-Learning

This notebook produces runnable evidence for three claims about Q-learning.
Each experiment generates output you can show an interviewer to back up what you say.

**Claims we will test:**
1. Q-learning convergence slows as state space grows — episodes to converge scales with grid size
2. The max operator in Q-learning causes systematic overestimation of Q-values — Double Q-learning eliminates this bias
3. Q-learning can learn the optimal policy from data collected by a purely random policy (off-policy), while SARSA cannot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import defaultdict

print("Imports ready.")
print(f"NumPy version: {np.__version__}")

## Shared Environment: Variable-Size GridWorld

All three experiments use a simple grid world with configurable size.
The agent starts at (0, 0) and must reach the goal at (size-1, size-1).
Every step costs -1 and reaching the goal gives +10.

```
┌───┬───┬───┐
│ S │   │   │   S = Start (0,0)
├───┼───┼───┤
│   │   │   │
├───┼───┼───┤
│   │   │ G │   G = Goal (size-1, size-1)
└───┴───┴───┘
```

Deterministic transitions. Actions: UP=0, RIGHT=1, DOWN=2, LEFT=3.

In [ ]:
class GridWorld:
    """
    Simple NxN grid world with deterministic transitions.

    Agent starts at (0, 0). Goal is at (size-1, size-1).
    Reward: -1 per step, +10 at goal.
    Actions: 0=UP, 1=RIGHT, 2=DOWN, 3=LEFT.
    """
    def __init__(self, size=5):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.n_actions = 4
        self.reset()

    def reset(self):
        self.pos = (0, 0)
        return self.pos

    def step(self, action):
        row, col = self.pos
        if action == 0:    # UP
            row = max(0, row - 1)
        elif action == 1:  # RIGHT
            col = min(self.size - 1, col + 1)
        elif action == 2:  # DOWN
            row = min(self.size - 1, row + 1)
        elif action == 3:  # LEFT
            col = max(0, col - 1)
        self.pos = (row, col)
        done = (self.pos == self.goal)
        reward = 10.0 if done else -1.0
        return self.pos, reward, done


# Verify the environment works
env = GridWorld(size=5)
np.random.seed(0)
state = env.reset()
total_reward = 0.0
steps = 0
for _ in range(500):
    action = np.random.randint(4)
    state, reward, done = env.step(action)
    total_reward += reward
    steps += 1
    if done:
        break

print(f"Test episode: {steps} steps, total reward: {total_reward:.0f}")
print(f"Final state: {state}, done: {done}")
print("GridWorld is ready.")

---

## Experiment 1: Complexity Benchmark — Convergence Rate vs State Space Size

**Claim:** Q-learning convergence slows as the state space grows. The number of
episodes needed to reach a performance threshold increases with grid size because
Q-learning must visit every (state, action) pair enough times to converge.
With |S| = N² states and |A| = 4 actions, total entries in the Q-table grow as
O(N²), and the agent must explore all of them.

**Why it matters in an interview:** When someone asks about Q-learning scalability,
you need to explain why tabular Q-learning breaks down in large state spaces.
This experiment gives you real numbers: a 10x10 grid already takes significantly
more episodes than a 3x3 grid, motivating the transition to function approximation
(DQN).

**What we will do:**
1. Run Q-learning on grids of size 3x3, 5x5, 7x7, and 10x10
2. Measure how many episodes it takes to consistently reach good performance
3. Plot episodes-to-converge vs state space size to show the scaling behavior

In [ ]:
def q_learning_train(env, n_episodes, alpha=0.1, gamma=0.99, epsilon=0.1,
                     max_steps_per_episode=500):
    """
    Train Q-learning and return the reward history.
    """
    Q = defaultdict(lambda: np.zeros(4))
    rewards_history = []

    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0.0

        for step in range(max_steps_per_episode):
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(4)
            else:
                action = np.argmax(Q[state])

            next_state, reward, done = env.step(action)
            total_reward += reward

            # Q-learning update: uses max Q(s', a')
            best_next_q = np.max(Q[next_state]) if not done else 0.0
            td_target = reward + gamma * best_next_q
            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error

            state = next_state
            if done:
                break

        rewards_history.append(total_reward)

    return Q, rewards_history


def episodes_to_converge(rewards_history, threshold_fraction=0.8,
                         optimal_reward=None, window=100):
    """
    Find the first episode where the smoothed reward exceeds
    threshold_fraction * optimal_reward for the rest of training.

    Returns the episode number, or len(rewards_history) if never reached.
    """
    target = threshold_fraction * optimal_reward
    n = len(rewards_history)
    if n < window:
        return n

    # Compute rolling average
    smoothed = np.convolve(rewards_history, np.ones(window) / window, mode='valid')

    for i, val in enumerate(smoothed):
        if val >= target:
            return i + window  # episode index where we crossed

    return n  # never converged


print("Q-learning training function ready.")
print("Convergence measurement function ready.")

In [ ]:
# Run Q-learning on grids of increasing size
grid_sizes = [3, 5, 7, 10]
n_seeds = 10
n_episodes = 8000  # enough for the larger grids

# Optimal reward for each grid: (size-1)*2 steps at -1, plus +10 at goal
# Optimal path from (0,0) to (size-1,size-1) takes (size-1)*2 steps
optimal_rewards = {s: 10.0 - (s - 1) * 2 for s in grid_sizes}

convergence_results = {s: [] for s in grid_sizes}
all_reward_histories = {s: [] for s in grid_sizes}

print(f"Running Q-learning on {len(grid_sizes)} grid sizes x {n_seeds} seeds...")
print(f"{'Size':>6} {'States':>8} {'Q-entries':>10} {'Optimal R':>10}")
print("-" * 40)
for size in grid_sizes:
    n_states = size * size
    n_q_entries = n_states * 4
    print(f"{size}x{size:>3} {n_states:>8} {n_q_entries:>10} {optimal_rewards[size]:>10.0f}")

print()

t0 = time.time()
for size in grid_sizes:
    for seed in range(n_seeds):
        np.random.seed(seed * 42 + size)
        env = GridWorld(size=size)
        _, rewards = q_learning_train(env, n_episodes=n_episodes,
                                      alpha=0.1, gamma=0.99, epsilon=0.1,
                                      max_steps_per_episode=size * size * 10)
        all_reward_histories[size].append(rewards)

        # Measure episodes to converge
        ep_conv = episodes_to_converge(rewards, threshold_fraction=0.7,
                                       optimal_reward=optimal_rewards[size],
                                       window=100)
        convergence_results[size].append(ep_conv)

    mean_conv = np.mean(convergence_results[size])
    std_conv = np.std(convergence_results[size])
    print(f"  {size}x{size}: converged in {mean_conv:.0f} +/- {std_conv:.0f} episodes")

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Learning curves for each grid size
ax1 = axes[0]
colors = ['#4caf50', '#2196f3', '#ff9800', '#f44336']
window = 100

for size, color in zip(grid_sizes, colors):
    # Average across seeds
    avg_rewards = np.mean(all_reward_histories[size], axis=0)
    smoothed = np.convolve(avg_rewards, np.ones(window) / window, mode='valid')
    ax1.plot(range(window - 1, len(avg_rewards)), smoothed,
             color=color, linewidth=2, label=f'{size}x{size} ({size*size} states)')

ax1.set_xlabel('Episode', fontsize=13)
ax1.set_ylabel('Total Reward (smoothed)', fontsize=13)
ax1.set_title('Q-Learning Convergence by Grid Size', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: Episodes to converge vs state space size
ax2 = axes[1]
state_space_sizes = [s * s for s in grid_sizes]
mean_convergences = [np.mean(convergence_results[s]) for s in grid_sizes]
std_convergences = [np.std(convergence_results[s]) for s in grid_sizes]

ax2.errorbar(state_space_sizes, mean_convergences, yerr=std_convergences,
             fmt='o-', color='#d32f2f', linewidth=2.5, markersize=10,
             capsize=8, capthick=2, label='Measured')

# Fit a line to show scaling trend
coeffs = np.polyfit(state_space_sizes, mean_convergences, 1)
fit_x = np.linspace(min(state_space_sizes), max(state_space_sizes), 100)
fit_y = np.polyval(coeffs, fit_x)
ax2.plot(fit_x, fit_y, '--', color='gray', linewidth=1.5, alpha=0.7,
         label=f'Linear fit (slope={coeffs[0]:.1f})')

ax2.set_xlabel('State Space Size (|S| = N\u00b2)', fontsize=13)
ax2.set_ylabel('Episodes to Converge', fontsize=13)
ax2.set_title('Scaling: More States = Slower Convergence', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Add grid size labels to data points
for size, ss, mc in zip(grid_sizes, state_space_sizes, mean_convergences):
    ax2.annotate(f'{size}x{size}', (ss, mc), textcoords='offset points',
                 xytext=(10, 10), fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print("CONVERGENCE SCALING RESULTS")
print("=" * 60)
print(f"{'Grid':>8} {'|S|':>6} {'|S|x|A|':>8} {'Episodes':>12} {'Ratio':>8}")
print("-" * 60)
baseline = mean_convergences[0]
for size, ss, mc in zip(grid_sizes, state_space_sizes, mean_convergences):
    ratio = mc / baseline
    print(f"{size}x{size:>3} {ss:>6} {ss*4:>8} {mc:>12.0f} {ratio:>8.1f}x")

print()
print(f"Scaling from {grid_sizes[0]}x{grid_sizes[0]} to {grid_sizes[-1]}x{grid_sizes[-1]}:")
state_ratio = state_space_sizes[-1] / state_space_sizes[0]
episode_ratio = mean_convergences[-1] / mean_convergences[0]
print(f"  State space grew {state_ratio:.1f}x")
print(f"  Episodes to converge grew {episode_ratio:.1f}x")
print()
print('Interview sentence: "Tabular Q-learning must visit every (s,a) pair')
print(f'  enough times to converge. Going from a {grid_sizes[0]}x{grid_sizes[0]} grid ({state_space_sizes[0]} states)')
print(f'  to {grid_sizes[-1]}x{grid_sizes[-1]} ({state_space_sizes[-1]} states), convergence time grew')
print(f'  {episode_ratio:.1f}x. This is why DQN replaces the table with a neural')
print(f'  network \u2014 function approximation generalizes across similar states."')

### What we just saw

Two things are visible in the plots:

1. **Smaller grids converge faster.** The 3x3 grid reaches good performance quickly
   because there are only 9 states (36 Q-table entries). The 10x10 grid has 100
   states (400 Q-table entries) and takes much longer.

2. **The relationship is roughly linear.** Episodes to converge scales approximately
   linearly with the number of states, because the agent must explore each state-action
   pair enough times.

This is the fundamental scalability bottleneck of tabular RL. Atari games have
roughly 10^{70} possible screen states. A Q-table for that is impossible.
This experiment gives you the concrete numbers to explain why DQN was necessary.

---

## Experiment 2: Failure Mode — Maximization Bias (Overestimation)

**Claim:** Q-learning systematically overestimates Q-values because of the max
operator. When Q estimates are noisy (which they always are during learning),
`E[max Q̂(s,a)] >= max E[Q̂(s,a)]` by Jensen's inequality. This makes Q-learning
think actions are better than they actually are. Double Q-learning fixes this
by decoupling action selection from evaluation.

**Why it matters in an interview:** Overestimation bias is the single most
important failure mode of Q-learning. It led to Double DQN, which improved
Atari scores on 80% of games. Being able to demonstrate the bias with real
numbers — and show that Double Q-learning eliminates it — signals deep
understanding of why DQN improvements exist.

**What we will do:**
1. Create a simple environment where the true Q-values are known exactly
2. Run standard Q-learning and track how learned Q-values compare to true values
3. Run Double Q-learning and show it eliminates the overestimation

In [ ]:
class MaxBiasEnv:
    """
    A simple environment designed to expose maximization bias.

    The environment has 3 states: A (start), B, and terminal.

         A ---right---> B ---[10 actions]---> terminal
         |                   each gives reward ~ N(0, 1)
         left
         |
         v
         terminal (reward = 0)

    From state A:
      - LEFT: go to terminal, reward = 0 (always)
      - RIGHT: go to state B, reward = 0

    From state B:
      - 10 actions, each gives reward ~ N(mean=0, std=1)
      - All actions lead to terminal

    TRUE Q-values:
      Q*(A, left) = 0    (get 0 and done)
      Q*(A, right) = 0   (get 0 to B, then E[max N(0,1)] from B... but see below)
      Q*(B, any) = 0     (each reward has mean 0)

    The TRAP: In state B, each action's reward has mean 0. So the true
    value of going right is 0. But Q-learning estimates Q(B, a) with noise.
    Taking max over 10 noisy estimates with mean 0 gives a POSITIVE number.
    This makes Q-learning think RIGHT is better than LEFT at state A.

    With gamma=1: Q*(A, right) = 0 + gamma * max_a Q*(B, a) = 0 + 0 = 0.
    But Q-learning will estimate Q(A, right) > 0 due to maximization bias.
    """

    def __init__(self, n_b_actions=10, reward_std=1.0):
        self.n_b_actions = n_b_actions
        self.reward_std = reward_std
        self.state = 'A'

    def reset(self):
        self.state = 'A'
        return self.state

    def get_n_actions(self, state):
        if state == 'A':
            return 2  # LEFT=0, RIGHT=1
        elif state == 'B':
            return self.n_b_actions
        else:
            return 0

    def step(self, action):
        if self.state == 'A':
            if action == 0:  # LEFT -> terminal with reward 0
                self.state = 'terminal'
                return self.state, 0.0, True
            else:  # RIGHT -> go to B with reward 0
                self.state = 'B'
                return self.state, 0.0, False
        elif self.state == 'B':
            # Any action: reward ~ N(0, std), go to terminal
            reward = np.random.normal(0.0, self.reward_std)
            self.state = 'terminal'
            return self.state, reward, True
        else:
            return self.state, 0.0, True


# Verify the environment
env = MaxBiasEnv(n_b_actions=10)
print("MaxBias Environment:")
print("  State A: 2 actions (LEFT=terminal r=0, RIGHT=B r=0)")
print(f"  State B: {env.n_b_actions} actions (each -> terminal, reward ~ N(0, {env.reward_std}))")
print(f"  True Q*(A, left) = 0")
print(f"  True Q*(A, right) = 0")
print(f"  True Q*(B, any action) = 0")
print()

# Verify: sample many B-action rewards to confirm mean ~= 0
b_rewards = [np.random.normal(0.0, 1.0) for _ in range(10000)]
print(f"Empirical check: mean of B rewards = {np.mean(b_rewards):.4f} (should be ~0)")
print(f"  max of 10 B rewards sampled once = {max(np.random.normal(0.0, 1.0) for _ in range(10)):.4f}")
print(f"  (This max is positive due to noise — that is the bias!)")

In [ ]:
def q_learning_maxbias(env, n_episodes=1000, alpha=0.1, gamma=1.0, epsilon=0.1):
    """
    Standard Q-learning on the MaxBias environment.
    Track Q(A, left) and Q(A, right) over time, and the fraction
    of times the agent chooses LEFT at state A.
    """
    # Q-values: Q_A[action] for state A, Q_B[action] for state B
    Q_A = np.zeros(2)        # LEFT, RIGHT
    Q_B = np.zeros(env.n_b_actions)

    q_a_left_history = []
    q_a_right_history = []
    left_fraction_history = []
    left_count = 0

    for episode in range(n_episodes):
        state = env.reset()

        while state != 'terminal':
            if state == 'A':
                # Epsilon-greedy over 2 actions
                if np.random.random() < epsilon:
                    action = np.random.randint(2)
                else:
                    action = np.argmax(Q_A)

                if action == 0:
                    left_count += 1

                next_state, reward, done = env.step(action)

                if done:
                    # Terminal: Q-target = reward
                    td_target = reward
                else:
                    # Next state is B: use max Q(B, a)
                    td_target = reward + gamma * np.max(Q_B)

                Q_A[action] += alpha * (td_target - Q_A[action])
                state = next_state

            elif state == 'B':
                # Epsilon-greedy over n_b_actions
                if np.random.random() < epsilon:
                    action = np.random.randint(env.n_b_actions)
                else:
                    action = np.argmax(Q_B)

                next_state, reward, done = env.step(action)

                td_target = reward  # terminal
                Q_B[action] += alpha * (td_target - Q_B[action])
                state = next_state

        q_a_left_history.append(Q_A[0])
        q_a_right_history.append(Q_A[1])
        left_fraction_history.append(left_count / (episode + 1))

    return q_a_left_history, q_a_right_history, left_fraction_history


def double_q_learning_maxbias(env, n_episodes=1000, alpha=0.1, gamma=1.0,
                               epsilon=0.1):
    """
    Double Q-learning on the MaxBias environment.
    Uses two Q-tables. To update Q1:
      a* = argmax_a Q1(s', a)   <- Q1 selects
      target = r + gamma * Q2(s', a*)  <- Q2 evaluates
    Randomly choose which table to update each step.
    """
    Q1_A = np.zeros(2)
    Q1_B = np.zeros(env.n_b_actions)
    Q2_A = np.zeros(2)
    Q2_B = np.zeros(env.n_b_actions)

    q_a_left_history = []
    q_a_right_history = []
    left_fraction_history = []
    left_count = 0

    for episode in range(n_episodes):
        state = env.reset()

        while state != 'terminal':
            if state == 'A':
                # Epsilon-greedy using sum of both Q-tables
                q_sum = Q1_A + Q2_A
                if np.random.random() < epsilon:
                    action = np.random.randint(2)
                else:
                    action = np.argmax(q_sum)

                if action == 0:
                    left_count += 1

                next_state, reward, done = env.step(action)

                # Randomly update Q1 or Q2
                if np.random.random() < 0.5:
                    # Update Q1
                    if done:
                        td_target = reward
                    else:
                        # Q1 selects action, Q2 evaluates
                        a_star = np.argmax(Q1_B)
                        td_target = reward + gamma * Q2_B[a_star]
                    Q1_A[action] += alpha * (td_target - Q1_A[action])
                else:
                    # Update Q2
                    if done:
                        td_target = reward
                    else:
                        # Q2 selects action, Q1 evaluates
                        a_star = np.argmax(Q2_B)
                        td_target = reward + gamma * Q1_B[a_star]
                    Q2_A[action] += alpha * (td_target - Q2_A[action])

                state = next_state

            elif state == 'B':
                q_sum_b = Q1_B + Q2_B
                if np.random.random() < epsilon:
                    action = np.random.randint(env.n_b_actions)
                else:
                    action = np.argmax(q_sum_b)

                next_state, reward, done = env.step(action)

                # Terminal, so target = reward
                if np.random.random() < 0.5:
                    Q1_B[action] += alpha * (reward - Q1_B[action])
                else:
                    Q2_B[action] += alpha * (reward - Q2_B[action])

                state = next_state

        # Average of both tables for reporting
        avg_left = (Q1_A[0] + Q2_A[0]) / 2
        avg_right = (Q1_A[1] + Q2_A[1]) / 2
        q_a_left_history.append(avg_left)
        q_a_right_history.append(avg_right)
        left_fraction_history.append(left_count / (episode + 1))

    return q_a_left_history, q_a_right_history, left_fraction_history


print("Standard Q-learning and Double Q-learning implementations ready.")

In [ ]:
# Run both algorithms many times to get stable results
n_episodes = 1000
n_seeds = 30

ql_left_all = []
ql_right_all = []
ql_leftfrac_all = []
dql_left_all = []
dql_right_all = []
dql_leftfrac_all = []

print(f"Running {n_seeds} seeds each for Q-learning and Double Q-learning...")
t0 = time.time()

for seed in range(n_seeds):
    np.random.seed(seed * 7)
    env = MaxBiasEnv(n_b_actions=10)
    ql_left, ql_right, ql_frac = q_learning_maxbias(env, n_episodes,
                                                     alpha=0.1, gamma=1.0,
                                                     epsilon=0.1)
    ql_left_all.append(ql_left)
    ql_right_all.append(ql_right)
    ql_leftfrac_all.append(ql_frac)

    np.random.seed(seed * 7)
    env = MaxBiasEnv(n_b_actions=10)
    dql_left, dql_right, dql_frac = double_q_learning_maxbias(env, n_episodes,
                                                                alpha=0.1, gamma=1.0,
                                                                epsilon=0.1)
    dql_left_all.append(dql_left)
    dql_right_all.append(dql_right)
    dql_leftfrac_all.append(dql_frac)

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")

# Report final Q-values
ql_final_left = np.mean([h[-1] for h in ql_left_all])
ql_final_right = np.mean([h[-1] for h in ql_right_all])
dql_final_left = np.mean([h[-1] for h in dql_left_all])
dql_final_right = np.mean([h[-1] for h in dql_right_all])

print(f"\nFinal Q(A, left)  — True value: 0.00")
print(f"  Q-learning:       {ql_final_left:+.4f}")
print(f"  Double Q-learning:{dql_final_left:+.4f}")
print(f"\nFinal Q(A, right) — True value: 0.00")
print(f"  Q-learning:       {ql_final_right:+.4f}  {'<-- OVERESTIMATED' if ql_final_right > 0.1 else ''}")
print(f"  Double Q-learning:{dql_final_right:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Q(A, right) over training — shows overestimation
ax1 = axes[0]
window = 50

ql_right_avg = np.mean(ql_right_all, axis=0)
dql_right_avg = np.mean(dql_right_all, axis=0)

ql_smoothed = np.convolve(ql_right_avg, np.ones(window) / window, mode='valid')
dql_smoothed = np.convolve(dql_right_avg, np.ones(window) / window, mode='valid')

ax1.plot(range(window - 1, len(ql_right_avg)), ql_smoothed,
         color='#f44336', linewidth=2.5, label='Q-learning')
ax1.plot(range(window - 1, len(dql_right_avg)), dql_smoothed,
         color='#2196f3', linewidth=2.5, label='Double Q-learning')
ax1.axhline(y=0.0, color='black', linewidth=1.5, linestyle='--',
            label='True Q*(A, right) = 0')

ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Q(A, right)', fontsize=12)
ax1.set_title('Q(A, right) Over Training\n(True value = 0)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Middle: Fraction of LEFT actions at A — optimal is 50% (both are equal)
ax2 = axes[1]

ql_frac_avg = np.mean(ql_leftfrac_all, axis=0)
dql_frac_avg = np.mean(dql_leftfrac_all, axis=0)

ax2.plot(ql_frac_avg, color='#f44336', linewidth=2.5, label='Q-learning')
ax2.plot(dql_frac_avg, color='#2196f3', linewidth=2.5, label='Double Q-learning')
ax2.axhline(y=0.5, color='black', linewidth=1.5, linestyle='--',
            label='Optimal (50%: both are equal)')

ax2.set_xlabel('Episode', fontsize=12)
ax2.set_ylabel('Fraction choosing LEFT at A', fontsize=12)
ax2.set_title('Action Selection at State A\n(Optimal: 50% LEFT)', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Right: Bar chart of final Q-values vs true values
ax3 = axes[2]
labels = ['Q(A,left)', 'Q(A,right)']
true_vals = [0.0, 0.0]
ql_vals = [ql_final_left, ql_final_right]
dql_vals = [dql_final_left, dql_final_right]

x = np.arange(len(labels))
width = 0.25

bars1 = ax3.bar(x - width, true_vals, width, label='True Q*', color='#4caf50',
                edgecolor='black', linewidth=1.2)
bars2 = ax3.bar(x, ql_vals, width, label='Q-learning', color='#f44336',
                edgecolor='black', linewidth=1.2)
bars3 = ax3.bar(x + width, dql_vals, width, label='Double Q-learning',
                color='#2196f3', edgecolor='black', linewidth=1.2)

ax3.set_xticks(x)
ax3.set_xticklabels(labels, fontsize=12)
ax3.set_ylabel('Q-value', fontsize=12)
ax3.set_title('Final Q-values vs True Values\n(Closer to green = better)', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')
ax3.axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# Quantify the overestimation
overestimation = ql_final_right - 0.0
double_error = abs(dql_final_right - 0.0)

print("MAXIMIZATION BIAS RESULTS")
print("=" * 60)
print(f"True Q*(A, right) = 0.00")
print(f"Q-learning estimate:        {ql_final_right:+.4f}  (overestimates by {overestimation:.4f})")
print(f"Double Q-learning estimate: {dql_final_right:+.4f}  (error = {double_error:.4f})")
print()
ql_left_pct = ql_frac_avg[-1] * 100
dql_left_pct = dql_frac_avg[-1] * 100
print(f"LEFT action percentage (should be ~50%):")
print(f"  Q-learning:        {ql_left_pct:.1f}%  {'<-- biased toward RIGHT' if ql_left_pct < 40 else ''}")
print(f"  Double Q-learning: {dql_left_pct:.1f}%")
print()
print('Interview sentence: "Q-learning\'s max operator overestimates Q-values')
print('  because E[max Q̂] >= max E[Q̂] by Jensen\'s inequality. In my experiment,')
print(f'  Q-learning estimated Q(A, right) = {ql_final_right:+.2f} when the true value is 0.')
print(f'  This made the agent choose RIGHT {100 - ql_left_pct:.0f}% of the time instead of 50%.')
print(f'  Double Q-learning decouples selection and evaluation, reducing the')
print(f'  estimate to {dql_final_right:+.2f} and choosing LEFT {dql_left_pct:.0f}% of the time."')

### What we just saw

Three clear results:

1. **Q-learning overestimates Q(A, right).** The true value is 0, but Q-learning
   learns a positive value. This happens because in state B, the agent has 10 actions
   each with reward ~ N(0, 1). Taking the max of 10 noisy estimates with mean 0
   gives a positive number. This inflated Q(B) propagates to Q(A, right).

2. **The bias affects action selection.** Because Q(A, right) is overestimated,
   Q-learning chooses RIGHT more often than it should. Both actions are equally
   good (true value = 0), so the agent should choose LEFT about 50% of the time.
   Q-learning is biased toward RIGHT.

3. **Double Q-learning fixes the bias.** By using one table to select the best
   action and a second table to evaluate it, the noise in selection does not
   correlate with the noise in evaluation. The expected value of the estimate
   returns to approximately 0.

**Why this matters in deep RL:** In DQN, Q-values are estimated by a neural network
with significant approximation error. The max over thousands of state-action pairs
creates large overestimation. Double DQN uses the online network to select actions
and the target network to evaluate them — the same idea, zero extra parameters.

---

## Experiment 3: Ablation — Off-Policy Advantage: Learning from Random Data

**Claim:** Q-learning can learn the optimal policy from data collected by a
purely random behavior policy (off-policy property). SARSA cannot do this
effectively because it is on-policy: it learns the value of the policy that
collected the data, not the optimal policy.

**Why it matters in an interview:** Off-policy learning is Q-learning's most
important practical advantage. It enables experience replay (reuse old data),
learning from demonstrations (human data), and offline RL (learn from logged
data without further interaction). Showing that Q-learning succeeds where SARSA
fails on the same random data demonstrates this concretely.

**What we will do:**
1. Collect a large batch of transitions from a purely random policy
2. Train Q-learning on this batch (offline, off-policy)
3. Train SARSA on this batch (offline, on-policy)
4. Evaluate both learned policies and compare

In [ ]:
def collect_random_transitions(env, n_episodes=5000):
    """
    Collect transitions from a purely random policy.

    Returns a list of (state, action, reward, next_state, done) tuples.
    Also returns SARSA-style tuples: (state, action, reward, next_state, next_action).
    """
    transitions = []
    sarsa_transitions = []

    for ep in range(n_episodes):
        state = env.reset()
        action = np.random.randint(4)  # random first action

        for step in range(env.size * env.size * 10):
            next_state, reward, done = env.step(action)
            next_action = np.random.randint(4)  # random next action

            transitions.append((state, action, reward, next_state, done))
            sarsa_transitions.append((state, action, reward, next_state,
                                      next_action, done))

            state = next_state
            action = next_action

            if done:
                break

    return transitions, sarsa_transitions


# Collect random data from a 5x5 grid
grid_size = 5
np.random.seed(42)
env = GridWorld(size=grid_size)

print(f"Collecting random transitions from {grid_size}x{grid_size} GridWorld...")
t0 = time.time()
transitions, sarsa_transitions = collect_random_transitions(env, n_episodes=5000)
elapsed = time.time() - t0

print(f"Collected {len(transitions)} transitions in {elapsed:.1f}s")
print(f"  (from a purely random policy — no learning during collection)")
print(f"  Unique states visited: {len(set(t[0] for t in transitions))}")
print(f"  Transitions reaching goal: {sum(1 for t in transitions if t[4])}")

In [ ]:
def batch_q_learning(transitions, n_passes=30, alpha=0.1, gamma=0.99):
    """
    Train Q-learning on a fixed batch of transitions (offline, off-policy).
    Iterate over the batch multiple times (passes) to converge.
    """
    Q = defaultdict(lambda: np.zeros(4))
    q_history = []  # track max Q at start state per pass

    for pass_num in range(n_passes):
        # Shuffle transitions each pass for stability
        np.random.shuffle(transitions)

        for state, action, reward, next_state, done in transitions:
            if done:
                td_target = reward
            else:
                td_target = reward + gamma * np.max(Q[next_state])

            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error

        q_history.append(np.max(Q[(0, 0)]))

    return dict(Q), q_history


def batch_sarsa(sarsa_transitions, n_passes=30, alpha=0.1, gamma=0.99):
    """
    Train SARSA on a fixed batch of transitions (offline, on-policy).
    SARSA uses the next action that was actually taken (by the random policy).
    """
    Q = defaultdict(lambda: np.zeros(4))
    q_history = []

    for pass_num in range(n_passes):
        np.random.shuffle(sarsa_transitions)

        for state, action, reward, next_state, next_action, done in sarsa_transitions:
            if done:
                td_target = reward
            else:
                # SARSA uses Q(s', a') where a' is the ACTUAL next action
                td_target = reward + gamma * Q[next_state][next_action]

            td_error = td_target - Q[state][action]
            Q[state][action] += alpha * td_error

        q_history.append(np.max(Q[(0, 0)]))

    return dict(Q), q_history


def evaluate_policy(Q, env, n_episodes=200, max_steps=200):
    """
    Evaluate a greedy policy derived from Q-values.
    Returns the average total reward.
    """
    total_rewards = []

    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0

        for step in range(max_steps):
            q_vals = Q.get(state, np.zeros(4))
            action = np.argmax(q_vals)
            state, reward, done = env.step(action)
            total_reward += reward
            if done:
                break

        total_rewards.append(total_reward)

    return np.mean(total_rewards), np.std(total_rewards)


print("Batch Q-learning and batch SARSA implementations ready.")
print("Policy evaluation function ready.")

In [ ]:
# Train both algorithms on the same random data
n_seeds = 10
n_passes = 30

ql_eval_rewards = []
sarsa_eval_rewards = []
ql_q_histories = []
sarsa_q_histories = []

print(f"Training Q-learning and SARSA on random data ({n_seeds} seeds)...")
print(f"  Data: {len(transitions)} transitions from random policy")
print(f"  Passes: {n_passes} iterations over the batch")
print()

optimal_reward = 10.0 - (grid_size - 1) * 2  # optimal path reward
print(f"Optimal reward for {grid_size}x{grid_size} grid: {optimal_reward:.0f}")
print()

for seed in range(n_seeds):
    np.random.seed(seed * 13)

    # Make copies so shuffling does not interfere between seeds
    trans_copy = list(transitions)
    sarsa_trans_copy = list(sarsa_transitions)

    # Train Q-learning
    Q_ql, ql_qhist = batch_q_learning(trans_copy, n_passes=n_passes,
                                       alpha=0.05, gamma=0.99)
    ql_q_histories.append(ql_qhist)

    # Train SARSA
    Q_sarsa, sarsa_qhist = batch_sarsa(sarsa_trans_copy, n_passes=n_passes,
                                        alpha=0.05, gamma=0.99)
    sarsa_q_histories.append(sarsa_qhist)

    # Evaluate both policies
    env_eval = GridWorld(size=grid_size)
    ql_mean, ql_std = evaluate_policy(Q_ql, env_eval)
    sarsa_mean, sarsa_std = evaluate_policy(Q_sarsa, env_eval)

    ql_eval_rewards.append(ql_mean)
    sarsa_eval_rewards.append(sarsa_mean)

    print(f"  Seed {seed}: Q-learning reward = {ql_mean:.1f}, "
          f"SARSA reward = {sarsa_mean:.1f}")

print()
print(f"Average policy quality (greedy evaluation):")
print(f"  Q-learning:  {np.mean(ql_eval_rewards):.1f} +/- {np.std(ql_eval_rewards):.1f}")
print(f"  SARSA:        {np.mean(sarsa_eval_rewards):.1f} +/- {np.std(sarsa_eval_rewards):.1f}")
print(f"  Optimal:      {optimal_reward:.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: Q(start) over training passes
ax1 = axes[0]
ql_q_avg = np.mean(ql_q_histories, axis=0)
sarsa_q_avg = np.mean(sarsa_q_histories, axis=0)

ax1.plot(range(1, n_passes + 1), ql_q_avg, color='#2196f3', linewidth=2.5,
         marker='o', markersize=4, label='Q-learning')
ax1.plot(range(1, n_passes + 1), sarsa_q_avg, color='#ff9800', linewidth=2.5,
         marker='s', markersize=4, label='SARSA')

ax1.set_xlabel('Pass over batch', fontsize=12)
ax1.set_ylabel('max Q(start state)', fontsize=12)
ax1.set_title('Q-value at Start State\nDuring Batch Training', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Middle: Evaluation reward comparison (bar chart)
ax2 = axes[1]
methods = ['Q-learning\n(off-policy)', 'SARSA\n(on-policy)', 'Optimal']
means = [np.mean(ql_eval_rewards), np.mean(sarsa_eval_rewards), optimal_reward]
stds = [np.std(ql_eval_rewards), np.std(sarsa_eval_rewards), 0]
bar_colors = ['#2196f3', '#ff9800', '#4caf50']

bars = ax2.bar(methods, means, yerr=stds, capsize=8, capthick=2,
               color=bar_colors, edgecolor='black', linewidth=1.2)

# Add value labels on bars
for bar, mean_val in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{mean_val:.1f}', ha='center', va='bottom', fontsize=12,
             fontweight='bold')

ax2.set_ylabel('Average Reward (greedy policy)', fontsize=12)
ax2.set_title('Policy Quality After Training\non Random Data', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Right: Visualize the learned policies on the grid
ax3 = axes[2]

# Use the last seed's Q-values for visualization
arrow_map = {0: (0, 0.35), 1: (0.35, 0), 2: (0, -0.35), 3: (-0.35, 0)}
action_labels = ['\u2191', '\u2192', '\u2193', '\u2190']

for row in range(grid_size):
    for col in range(grid_size):
        y = grid_size - 1 - row

        if (row, col) == (grid_size - 1, grid_size - 1):
            cell_color = '#c8e6c9'
        elif (row, col) == (0, 0):
            cell_color = '#bbdefb'
        else:
            cell_color = 'white'

        from matplotlib.patches import Rectangle
        rect = Rectangle((col, y), 1, 1, facecolor=cell_color,
                          edgecolor='black', linewidth=1)
        ax3.add_patch(rect)

        state = (row, col)
        if state != (grid_size - 1, grid_size - 1):
            # Q-learning action (blue arrow)
            ql_action = np.argmax(Q_ql.get(state, np.zeros(4)))
            dx_ql, dy_ql = arrow_map[ql_action]
            ax3.arrow(col + 0.3, y + 0.5, dx_ql * 0.5, dy_ql * 0.5,
                      head_width=0.1, head_length=0.05,
                      fc='#2196f3', ec='#2196f3', linewidth=1.5)

            # SARSA action (orange arrow)
            sarsa_action = np.argmax(Q_sarsa.get(state, np.zeros(4)))
            dx_s, dy_s = arrow_map[sarsa_action]
            ax3.arrow(col + 0.7, y + 0.5, dx_s * 0.5, dy_s * 0.5,
                      head_width=0.1, head_length=0.05,
                      fc='#ff9800', ec='#ff9800', linewidth=1.5)

ax3.set_xlim(0, grid_size)
ax3.set_ylim(0, grid_size)
ax3.set_aspect('equal')
ax3.axis('off')
ax3.set_title(f'Learned Policies on {grid_size}x{grid_size} Grid\n'
              f'Blue=Q-learning, Orange=SARSA', fontsize=13, fontweight='bold')

# Add legend
ax3.plot([], [], color='#2196f3', linewidth=3, label='Q-learning')
ax3.plot([], [], color='#ff9800', linewidth=3, label='SARSA')
ax3.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

# Print the key finding
ql_avg = np.mean(ql_eval_rewards)
sarsa_avg = np.mean(sarsa_eval_rewards)

print("OFF-POLICY vs ON-POLICY: LEARNING FROM RANDOM DATA")
print("=" * 60)
print(f"Data source: purely random policy (uniform random actions)")
print(f"Training: {n_passes} passes over {len(transitions)} transitions")
print(f"Evaluation: greedy policy derived from learned Q-values")
print()
print(f"{'Method':<20} {'Avg Reward':>12} {'vs Optimal':>12}")
print("-" * 50)
print(f"{'Q-learning':<20} {ql_avg:>12.1f} {ql_avg/optimal_reward*100:>11.0f}%")
print(f"{'SARSA':<20} {sarsa_avg:>12.1f} {sarsa_avg/optimal_reward*100:>11.0f}%")
print(f"{'Optimal':<20} {optimal_reward:>12.0f} {'100%':>12}")
print()
print("WHY SARSA DOES WORSE:")
print("  SARSA learns Q^\u03c0 (the value of the random policy that collected the data).")
print("  The random policy is terrible, so SARSA learns terrible Q-values.")
print("  When you extract a greedy policy from these Q-values, it is suboptimal.")
print()
print("WHY Q-LEARNING WORKS:")
print("  Q-learning learns Q* (the optimal Q-values) regardless of the data")
print("  collection policy. It uses max Q(s', a') in the update, so it always")
print("  estimates what the BEST action would give, not what the random policy gives.")
print()
print('Interview sentence: "Q-learning is off-policy: it learns the optimal')
print(f'  policy from any behavior policy. I trained Q-learning on {len(transitions)}')
print(f'  transitions from a random policy and the greedy policy achieved')
print(f'  {ql_avg:.0f} reward ({ql_avg/optimal_reward*100:.0f}% of optimal). SARSA only')
print(f'  achieved {sarsa_avg:.0f} because it learned the value of the random policy,')
print(f'  not the optimal one. This is why experience replay works in DQN \u2014')
print(f'  off-policy learning lets you reuse old data collected by earlier policies."')

### What we just saw

Three clear results:

1. **Q-learning learns a good policy from random data.** Even though the data
   was collected by a uniformly random policy, Q-learning extracted a near-optimal
   greedy policy. This is the off-policy property in action: the max operator in
   the update decouples the learned policy from the data collection policy.

2. **SARSA learns a worse policy from the same data.** SARSA uses Q(s', a') where
   a' is the actual next action taken by the random policy. Since the random policy
   is terrible, SARSA's Q-values reflect the value of being random — not the value
   of being optimal. The greedy policy derived from these Q-values is suboptimal.

3. **The policy arrows tell the story.** Q-learning's arrows point consistently
   toward the goal (down-right). SARSA's arrows may point in suboptimal directions
   because its Q-values were learned under the assumption that the agent will
   continue acting randomly.

**Why this matters for DQN:** Experience replay stores past transitions in a buffer.
As the policy improves, old data becomes "off-policy" (collected by a worse policy).
Q-learning can still learn from this old data because it is off-policy. SARSA
cannot — it would need to throw away the old data and only use transitions from
the current policy, which is much less sample-efficient.

---

## Summary: Claims Now Backed by Evidence

1. **Q-learning convergence slows as state space grows.** Going from a 3x3 grid
   (9 states) to a 10x10 grid (100 states), the number of episodes to converge
   increased significantly. This scaling is roughly linear in the number of states
   because every (state, action) pair must be visited enough times. This motivates
   function approximation (DQN). (Experiment 1)

2. **The max operator causes systematic overestimation (maximization bias).**
   In an environment where the true Q-values are 0, Q-learning consistently
   overestimated them due to E[max Q̂] >= max E[Q̂]. Double Q-learning decouples
   action selection and evaluation, eliminating the bias. This is the same idea
   behind Double DQN. (Experiment 2)

3. **Off-policy learning lets Q-learning learn from random data.** Q-learning
   extracted a near-optimal policy from transitions collected by a purely random
   policy. SARSA failed on the same data because it is on-policy: it learned the
   value of the random policy, not the optimal one. This property enables
   experience replay, learning from demonstrations, and offline RL. (Experiment 3)

For the full mathematical treatment and interview Q&A, see
[q-learning-interview.md](./q-learning-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)